# 01 · Blue Book residual-value fit

Fits a resale **retention curve** (resale ÷ price at age 1) per machine type on the Kaggle *Blue Book for Bulldozers* data (`TrainAndValid.csv`, ~412,000 US auction sales, 1989–2012).

Only the ratio is used in the tool. US dollar values never enter it.

**Model** (per product group, machine-model fixed effects):

$$\log(\text{real price}) = \alpha_{\text{model}} + b_1\,\text{age} + b_2\,\text{age}^2 + \varepsilon, \qquad \text{retention}(a) = e^{f(a) - f(1)}$$
arepsilon, \qquad 	ext{retention}(a) = e^{f(a) - f(1)}$$

Prices are deflated to 2012 dollars with US CPI-U. The plan also asked for a log(hours) term. 64% of rows have no hour-meter reading and another 18% read zero, so the main curve is age-only and the hours model is reported as a cross-check on the subset with readings.

**Data access:** log in to Kaggle, accept the competition rules, download `TrainAndValid.csv` and place it in `data/` (it is too large for GitHub, so it is git-ignored). In Colab, upload it or use the Kaggle API.

In [1]:
# Colab only: clone the repo so `core/` is importable (set REPO_URL to the team repo)
import os, sys
if "google.colab" in sys.modules and not os.path.exists("core"):
    REPO_URL = "https://github.com/<team>/<repo>.git"
    !git clone $REPO_URL repo
    os.chdir("repo")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [2]:
import pandas as pd
import plotly.graph_objects as go
from core.residual import fit_all, load_bluebook, CURVE_PATH, POINTS_PATH, GROUP_NAMES

df = load_bluebook()
print(f"{len(df):,} usable sales (YearMade known, age 0-25)")
df.groupby("ProductGroup").agg(sales=("SalePrice", "size"), models=("ModelID", "nunique"),
                               median_age=("age", "median"), share_with_hours=("log_hours", lambda s: s.notna().mean()))

353,165 usable sales (YearMade known, age 0-25)


,sales,models,median_age,share_with_hours
ProductGroup,,,,
BL,76109,346,7.0,0.163529
MG,18039,287,13.0,0.115361
SSL,40751,448,5.0,0.266816
TEX,90666,2026,7.0,0.217524
TTT,67798,709,9.0,0.176495
WL,59802,936,9.0,0.164627


In [3]:
curve, summary, points = fit_all()
curve.to_csv(CURVE_PATH, index=False, float_format="%.4f")
summary.to_csv(CURVE_PATH.parent / "residual_fit_summary.csv", index=False, float_format="%.5g")
points.to_csv(POINTS_PATH, index=False, float_format="%.4f")
summary.set_index("group").T

group,BL,MG,SSL,TEX,TTT,WL
group_name,Backhoe Loaders,Motor Graders,Skid Steer Loaders,Track Excavators,Track Type Tractors,Wheel Loader
b_age,-0.088329,-0.079816,-0.098421,-0.118318,-0.128699,-0.114119
se_age,0.000592,0.001251,0.000919,0.000812,0.00072,0.000806
b_age2,0.001094,0.001089,0.002353,0.001245,0.002451,0.001826
se_age2,0.000027,0.000044,0.000049,0.000037,0.000028,0.00003
r2_within,0.607247,0.564482,0.425293,0.565359,0.586545,0.559717
n,76109,18039,40751,90666,67798,59802
n_models,346,287,448,2026,709,936
hours_b_age,-0.054622,-0.054266,-0.065481,-0.072157,-0.126399,-0.097942
hours_b_age2,-0.000195,0.000264,0.001076,-0.000028,0.002423,0.001632


### Retention by age
Read the table at the tool's horizon. A backhoe (BL) at 9 years keeps about 54% of its age-1 price. The age-0 row is above 1 because the base is the age-1 price, not the new price.

In [4]:
curve.pivot(index="age", columns="group", values="retention").round(3)

group,BL,MG,SSL,TEX,TTT,WL
age,,,,,,
0,1.091,1.082,1.101,1.124,1.135,1.119
1,1.000,1.000,1.000,1.000,1.000,1.000
2,0.918,0.926,0.913,0.892,0.886,0.897
3,0.845,0.860,0.837,0.797,0.788,0.808
4,0.780,0.800,0.771,0.714,0.705,0.730
5,0.721,0.746,0.714,0.642,0.634,0.662
6,0.668,0.697,0.664,0.578,0.573,0.602
7,0.620,0.653,0.620,0.522,0.520,0.550
8,0.577,0.613,0.582,0.472,0.474,0.505


In [5]:
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
fig = go.Figure()
for color, (g, c) in zip(SERIES, curve.groupby("group")):
    fig.add_scatter(x=c["age"], y=c["retention"], name=GROUP_NAMES[g], mode="lines", line=dict(color=color, width=2))
fig.update_layout(title="Fitted retention curves", xaxis_title="Age (years)", yaxis_title="Resale ÷ age-1 price",
                  yaxis_tickformat=".0%", hovermode="x unified")
fig.show()

### Cross-check: hours model
`hours_b_log_hours` is the elasticity of resale price to meter hours, holding age fixed. A 10% increase in hours lowers price by about 10 × that coefficient, in percent. The age coefficients barely move when hours are added, which supports using the age-only curve.

### Caveats for the user manual
- US market, 1989–2012: the tool uses only ratios, and these may not carry over to Indian resale markets.
- Survivorship: machines that were scrapped never appear at auction, so old-age retention is biased upward.
- The quadratic is capped to be non-increasing beyond age 1.
- Tipper, transit mixer and crane have no Blue Book group, so they use a fixed salvage %.